# Part 1 - Vehicle Detection with a CNN Object Detector

**Objective:** build a deep-learning model that predicts the **type of vehicle** in an image and **localises it with a bounding box**.

Logic lives in `src/autodrive/detection/`. Each section maps to a task in the problem statement. Training is slow on CPU - on Colab/GPU set `train.device` to `0`.

### (Optional) Google Colab setup
Uncomment to run on a Colab GPU: upload/clone the repo and the data folder first.

In [ ]:
# !pip install -q ultralytics
# %cd /content/AutonomousDriving

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
from autodrive.common import load_config
from autodrive.common.plotting import show
from autodrive.detection import create_workspace
from autodrive.detection.workspace import tree
from autodrive.detection.model import resolve_device
cfg = load_config('configs/part1_detection.yaml')
# Quick experiment example: cfg = load_config('configs/part1_detection.yaml', ['train.epochs=5', 'train.fraction=0.25'])
print('device:', resolve_device(cfg['train']['device']))

## 1. Parent folder for custom model training + child folders

In [ ]:
ws = create_workspace(cfg)
print(tree(ws.root))

## 2. Prepare the dataset
* images are unzipped only if needed - in parallel and resumable (`dataset.extract_zip`)
* `labels.csv` covers 110k images; we keep the boxes of the 5,626 images we have
* degenerate boxes dropped, boxes clipped to image bounds
* image-level 80/10/10 split **stratified on the rarest class** in each image
* converted to YOLO format (`class cx cy w h`, normalised) + `data.yaml`

In [ ]:
from autodrive.detection.dataset import prepare_dataset
prep = prepare_dataset(cfg, ws)
prep['split_summary']

In [ ]:
prep['annotations'].head()

In [ ]:
print(tree(ws.dataset))
print(ws.data_yaml.read_text())

### Dataset EDA

In [ ]:
from autodrive.detection import eda
classes = cfg['dataset']['classes']
show(eda.fig_class_distribution(prep['annotations'], classes))
show(eda.fig_box_stats(prep['annotations']))
show(eda.fig_split_balance(prep['split_summary']))
show(eda.fig_samples(prep['annotations'], prep['images'], classes))

**Observations driving the model choice**
* 11 classes, heavily imbalanced (cars ~67% of boxes; motorcycle/bicycle ~100 each)
* many small objects (half of boxes < 0.8% of image area)
* fixed road cameras, day/night/yellow-tinted lighting, two resolutions (720x480, 342x228)

## 3. CNN architecture for object detection
**Chosen: YOLO11 (one-stage CNN detector), COCO-pretrained, fine-tuned.**

| Candidate | Speed | Small objects | Verdict |
|---|---|---|---|
| Custom CNN + regression head | fast | poor, single object | can't handle several vehicles per image |
| Faster R-CNN (two-stage) | slow | good | too slow for real-time traffic |
| SSD | fast | weak | anchor-based, struggles on small boxes |
| **YOLO11** | **real-time** | **good (3-scale PAN)** | **best accuracy/speed trade-off** |

Backbone: Conv-BN-SiLU + C3k2 CSP blocks + SPPF + C2PSA attention; Neck: PAN-FPN; Head: anchor-free decoupled cls/box (DFL + CIoU). Pretraining on COCO (which has car, bus, truck, person, bicycle, motorcycle) gives strong transfer for a 5.6k-image dataset. Augmentations (mosaic, mixup, HSV) address class imbalance and lighting.

In [ ]:
from autodrive.detection.model import build_model, describe_model
m = build_model(cfg)
print(describe_model(m))

In [ ]:
from autodrive.detection.model import train
model, results = train(cfg, ws)

In [ ]:
from autodrive.detection.evaluate import training_curves
run_dir = ws.runs / cfg['train']['run_name']
show(training_curves(run_dir))
display(Image(str(run_dir / 'results.png'), width=900))

## 4. Evaluate the model on the test split

In [ ]:
from autodrive.detection.evaluate import evaluate
ev = evaluate(cfg, ws)
ev['overall']

In [ ]:
ev['per_class']

In [ ]:
for f in ['confusion_matrix_normalized.png', 'BoxPR_curve.png']:
    p = ev['save_dir'] / f
    if p.exists(): display(Image(str(p), width=800))

## 5. Inference on sample images
Predictions vs ground truth, plus a vehicle count per image (vehicle-counting use case).

In [ ]:
from autodrive.detection.inference import predict
pr = predict(cfg, ws)
pr['counts']

In [ ]:
pr['sample_accuracy']

In [ ]:
display(Image(str(pr['comparison_figure']), width=1000))

Run on any new image or folder:

In [ ]:
# pr2 = predict(load_config('configs/part1_detection.yaml', ['predict.source=path/to/images']), ws)

## Conclusions
Fill in after full training: overall mAP@0.5 / mAP@0.5:0.95, strongest and weakest classes (rare classes such as motorcycle and non-motorized_vehicle are expected to be weakest), and whether vehicles in the sample images are detected accurately.